<a href="https://colab.research.google.com/github/DulaniDeSilva/Natural-Language-Processing-NLP-/blob/main/NaiveBayes_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Naive Bayes from Scratch


Classification algorithm based on Bayes' Theorem. "Given these features, what is the most probable class"?

We assume all features are independent of each other given the class. Pick the class gives the highest score.

Load and explore data

In [5]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Projects/NLP/spam.csv', encoding= "latin-1")

print(df.shape)
print(df.head())

(5572, 5)
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


Clean up the columns

In [6]:
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print(df['label'].value_counts())
df.head()

label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Preprocess text

In [7]:
import re

def preprocess(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]', '', text)
  tokens = text.split()
  return tokens

print(preprocess(df['text'][0]))

['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']


In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size =0.2, random_state = 42)

# Computing priors
total = len(train_df)
p_spam = (train_df['label']=='spam').sum()/ total
p_ham = (train_df['label'] == 'ham').sum()/total

print(f"P(spam) = {p_spam:.3f}")
print(f"P(ham) = {p_ham:.3f}")

P(spam) = 0.134
P(ham) = 0.866


Compute likelihoods P(word|class) with Laplace smoothing

For each class count how often each word appears

In [12]:
from collections import defaultdict

def build_word_counts(dataframe, label):
  word_counts = defaultdict(int)
  for message in dataframe[dataframe['label'] == label]['text']:
    for word in preprocess(message):
      word_counts[word] += 1
  return word_counts

spam_counts = build_word_counts(train_df, 'spam')
ham_counts = build_word_counts(train_df,'ham')

# building the vocabulary
vocabulary = set(spam_counts.keys()) | set(ham_counts.keys())
V = len(vocabulary)
print(f"Vocabulary size: {V}")

Vocabulary size: 7548


Compute likelihoods with Laplace smoothing

In [13]:
def compute_likelihood(word, word_counts, total_words, vocab_size):
  # Add 1 to numerator, V to denominator (Laplace smoothing)
  return (word_counts[word] +1) / (total_words + vocab_size)

spam_total = sum(spam_counts.values())
ham_total = sum(ham_counts.values())

Write the predict() funcction using log probabilities

Multiplying many small probabilites together causes underflow, so we use log probabilities and add instead of multiply

In [14]:
import math

def predict(message):
  tokens = preprocess(message)

  # start with log of prior
  log_prob_spam = math.log(p_spam)
  log_prob_ham = math.log(p_ham)

  # add log likelihood for each word
  for word in tokens:
    if word in vocabulary:
      log_prob_spam += math.log(compute_likelihood(word, spam_counts, spam_total, V))
      log_prob_ham += math.log(compute_likelihood(word, ham_counts, ham_total, V))

  return 'spam' if log_prob_spam > log_prob_ham else 'ham'


print(predict('WINNER!! This is the secret code to unlock the money: C3421.'))
print(predict('Sounds good, Tom, then see u there'))


spam
ham


Evaluate

In [19]:
test_df = test_df.copy()
test_df['predicted'] = test_df['text'].apply(predict)
accuracy = (test_df['label'] == test_df['predicted']).mean()
print(f"Accuracy: {accuracy:.3f}")

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(test_df['label'], test_df['predicted']))
print(classification_report(test_df['label'], test_df['predicted']))

Accuracy: 0.981
[[961   4]
 [ 17 133]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       965
        spam       0.97      0.89      0.93       150

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

